# KẺ MẠO DANH

Notebook baseline tạo trực tiếp `submission.zip` cho tập kiểm tra.

## 1. Cấu hình

In [ ]:
import os

# Phải đặt trước khi torch khởi tạo CUDA.
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

from pathlib import Path
import random
import time
import zipfile

import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageOps
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import v2

from torchvision import models

DATA_ROOT = Path('data')
SPLIT = 'public_test'        # đổi thành 'private_test' ở giai đoạn kiểm tra bí mật

IMAGE_SIZE = 224
BATCH_SIZE = 224
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Cố định kết quả giữa các lần chạy: cùng seed -> cùng file nộp.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[cấu hình] split={SPLIT} | batch={BATCH_SIZE} | ảnh={IMAGE_SIZE}px | seed={SEED}')
print(f'[cấu hình] thiết bị: {DEVICE}'
      + (f' ({torch.cuda.get_device_name(0)})' if DEVICE.type == 'cuda' else ''))

## 2. Mô hình và phép biến đổi ảnh

In [ ]:
backbone = models.efficientnet_b0(weights='DEFAULT')
dim = backbone.classifier[1].in_features
print(dim)
backbone.classifier[1] = nn.Linear(dim, 2)
backbone.classifier[0] = nn.Dropout(0.3)

In [ ]:
feature_modules = nn.Sequential(*list(backbone.children())[:-1])
head_module = list(backbone.children())[-1]

In [ ]:
total_params = sum(p.numel() for p in backbone.parameters())
print(total_params)

In [ ]:
# class CNN2(nn.Module):
#     """CNN hai khối tích chập, phân loại ảnh thật/giả."""

#     def __init__(self):
#         super().__init__()
#         self.features = nn.Sequential(
#             nn.Conv2d(3, 16, 3, padding=1),
#             nn.ReLU(),
#             nn.MaxPool2d(2),
#             nn.Conv2d(16, 32, 3, padding=1),
#             nn.ReLU(),
#             nn.MaxPool2d(2),
#         )
#         self.classifier = nn.Sequential(
#             nn.AdaptiveAvgPool2d(1),
#             nn.Flatten(),
#             nn.Linear(32, 2),
#         )

#     def forward(self, x):
#         return self.classifier(self.features(x))


train_tf = v2.Compose([
    v2.ToImage(),
    v2.Resize(256),
    v2.RandomResizedCrop((224, 224), scale=(0.75, 1.0), ratio=(0.9, 1.1)),
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomApply([
        v2.RandomAffine(degrees=8, translate=(0.05, 0.05), scale=(0.95, 1.05), shear=4)
    ], p=0.35),
    v2.RandomApply([v2.ColorJitter(0.2, 0.2, 0.2, 0.05)], p=0.8),
    v2.RandomApply([v2.GaussianBlur(kernel_size=5, sigma=(0.1, 1.5))], p=0.2),
    v2.RandomGrayscale(p=0.05),
    v2.ToDtype(torch.float, scale=True),
    v2.Normalize((.485, .456, .406), (.229, .224, .225)),
    v2.RandomErasing(p=0.2, scale=(0.02, 0.12), ratio=(0.5, 2.0), value='random')
])
test_tf = v2.Compose([
    v2.ToImage(),
    v2.Resize(256),
    v2.CenterCrop((224, 224)),
    v2.ToDtype(torch.float, scale=True),
    v2.Normalize((.485, .456, .406), (.229, .224, .225))
])

## 3. Dataset

In [ ]:
def load_image(path: Path, tf):
    with Image.open(path) as image:
        return tf(ImageOps.exif_transpose(image).convert('RGB'))


class SingleImages(Dataset):
    """Tách mỗi cặp train thành hai mẫu ảnh đơn kèm nhãn thật/giả."""

    def __init__(self, pairs, tf):
        self.tf = tf
        self.rows = []
        for row in pairs.itertuples(index=False):
            self.rows.append((row.image_0, int(row.fake_position == 0)))
            self.rows.append((row.image_1, int(row.fake_position == 1)))

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        relative, label = self.rows[index]
        return load_image(DATA_ROOT / 'train' / relative, self.tf), label


class ImagePairs(Dataset):
    """Trả về cả hai ảnh của một cặp để so sánh khi dự đoán."""

    def __init__(self, pairs):
        self.pairs = pairs.reset_index(drop=True)

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        row = self.pairs.iloc[index]
        first = load_image(DATA_ROOT / SPLIT / row.image_0, eval_tf)
        second = load_image(DATA_ROOT / SPLIT / row.image_1, eval_tf)
        return first, second, index


print('[dataset] đã định nghĩa SingleImages (train) và ImagePairs (dự đoán)')

## 4. Nạp dữ liệu

In [ ]:
from sklearn.model_selection import train_test_split
df = pd.read_csv(DATA_ROOT / 'train' / 'pairs.csv', dtype={'pair_id': str})
query = pd.read_csv(DATA_ROOT / SPLIT / 'pairs.csv', dtype={'pair_id': str})
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    stratify=df['fake_position']
)
train_data = SingleImages(train_df, train_tf)
val_data = SingleImages(val_df, test_tf)
BATCH_SIZE = 64
LOADER_KWARGS = {
    'batch_size': BATCH_SIZE,
    'pin_memory': True,
    'num_workers': 2,
    'persistent_workers': True,
}
train_loader = DataLoader(train_data, shuffle=True, **LOADER_KWARGS)
val_loader = DataLoader(val_data, shuffle=False, **LOADER_KWARGS)

print(f'[dữ liệu] train: {len(df):,} cặp -> {len(train_data):,} ảnh train')
print(f'[dữ liệu] {SPLIT}: {len(query):,} cặp cần dự đoán')
print(f'[dữ liệu] phân bố nhãn train: {train_data.fake_position.value_counts().to_dict()}')

In [ ]:
import matplotlib.pyplot as plt
imgs, labels = next(iter(train_loader))
fig, ax = plt.subplots(3, 4, figsize= (9, 12))
ax = ax.flatten()
mean = torch.tensor([0.485, 0.456, 0.406]).reshape(3, 1, 1)
std  = torch.tensor([0.229, 0.224, 0.225]).reshape(3, 1, 1)

for i in range(12):
    img, label = imgs[i], labels[i]
    img = img * std + mean
    img = img.permute(1,2,0).numpy()
    ax[i].imshow(img)
    ax[i].set_title(f"label {label}", fontsize=8)
    ax[i].axis('off')
plt.show()

## 5. Huấn luyện

In [ ]:
from tqdm import tqdm
from sklearn.metrics import precision_recall_fscore_support

In [ ]:
backbone = backbone.to(DEVICE)
optimizer = torch.optim.AdamW(
    [
        {'params': feature_modules.parameters(), 'lr': 3e-5},
        {'params': head_module.parameters(), 'lr': 3e-4},
    ],
    betas=(0.9, 0.999),
    weight_decay=1e-4,
)
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
best_f1 = -1.0
best_model_path = './best_model.pth'

In [ ]:
from torch.nn.utils import clip_grad_norm_

def soft_cross_entropy(logits, target_a, target_b, mix):
    return mix * criterion(logits, target_a) + (1.0 - mix) * criterion(logits, target_b)


def train_epoch(epoch, epochs, train_loader, scheduler):
    backbone.train()
    correct, count = 0, 0
    losses = []
    progress = tqdm(train_loader, desc=f'Train {epoch}/{epochs}')
    for images, labels in progress:
        images, labels = images.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        # MixUp/CutMix regularize only some batches; clean batches preserve the original labels.
        use_mix = random.random() < 0.5
        if use_mix:
            permutation = torch.randperm(images.size(0), device=DEVICE)
            mixed_labels = labels[permutation]
            mix = np.random.beta(0.4, 0.4)
            if random.random() < 0.5:
                images = mix * images + (1.0 - mix) * images[permutation]
            else:
                height, width = images.shape[-2:]
                cut_ratio = np.sqrt(1.0 - mix)
                cut_width, cut_height = int(width * cut_ratio), int(height * cut_ratio)
                center_x = random.randrange(width)
                center_y = random.randrange(height)
                x0 = max(center_x - cut_width // 2, 0)
                x1 = min(center_x + cut_width // 2, width)
                y0 = max(center_y - cut_height // 2, 0)
                y1 = min(center_y + cut_height // 2, height)
                images[:, :, y0:y1, x0:x1] = images[permutation, :, y0:y1, x0:x1]
                mix = 1.0 - ((x1 - x0) * (y1 - y0) / (width * height))
            outputs = backbone(images)
            loss = soft_cross_entropy(outputs, labels, mixed_labels, mix)
        else:
            outputs = backbone(images)
            loss = criterion(outputs, labels)

        loss.backward()
        clip_grad_norm_(backbone.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        losses.append(loss.item())
        pred = outputs.argmax(1)
        correct += (pred == labels).sum().item()
        count += labels.size(0)

    return correct / count, sum(losses) / len(losses)


def evaluate(epoch, epochs, val_loader):
    backbone.eval()
    correct, count = 0, 0
    losses = []
    preds, labels_all = [], []
    with torch.inference_mode():
        for images, labels in tqdm(val_loader, desc=f'Val {epoch}/{epochs}'):
            images, labels = images.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
            outputs = backbone(images)
            losses.append(criterion(outputs, labels).item())
            pred = outputs.argmax(1)
            correct += (pred == labels).sum().item()
            count += labels.size(0)
            preds.extend(pred.cpu().numpy())
            labels_all.extend(labels.cpu().numpy())
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels_all, preds, average='macro', zero_division=0
    )
    return correct / count, sum(losses) / len(losses), precision, recall, f1

In [ ]:
def train_model(epochs, train_loader, val_loader, patience=8):
    global best_f1
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=[3e-5, 3e-4],
        total_steps=epochs * len(train_loader),
        pct_start=0.15,
        anneal_strategy='cos',
        div_factor=10,
        final_div_factor=100,
    )
    wait = 0
    for epoch in range(1, epochs + 1):
        train_acc, train_loss = train_epoch(epoch, epochs, train_loader, scheduler)
        val_acc, val_loss, precision, recall, f1 = evaluate(epoch, epochs, val_loader)
        train_accs.append(train_acc)
        val_accs.append(val_acc)
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        f1s.append(f1)
        print(f'Train Acc: {train_acc:.4f}, Train loss: {train_loss:.4f}')
        print(f'Val Acc: {val_acc:.4f}, Val loss: {val_loss:.4f}')
        print(f'F1: {f1 * 100:.2f}% | Precision: {precision * 100:.2f}% | Recall: {recall * 100:.2f}%')

        if f1 > best_f1:
            best_f1 = f1
            torch.save(backbone.state_dict(), best_model_path)
            wait = 0
            print(f'Checkpoint at epoch {epoch}')
        else:
            wait += 1
            if wait >= patience:
                print(f'Early stopping at epoch {epoch}')
                break

In [ ]:
train_accs, train_losses, val_accs, val_losses, f1s = [], [], [], [], []
train_model(35, train_loader, val_loader, patience=8)

## 6. Dự đoán và đóng gói bài nộp

In [ ]:
started = time.time()
TTA_ENABLED = True
model = models.efficientnet_b0(weights=None)
model.classifier[0] = nn.Dropout(0.3)
model.classifier[1] = nn.Linear(dim, 2)
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE, weights_only=True))
model = model.to(DEVICE)
model.eval()
predictions = np.zeros(len(query), dtype=np.int64)
started = time.time()

with torch.inference_mode():
    for first, second, indices in DataLoader(ImagePairs(query), batch_size=BATCH_SIZE):
        first = first.to(DEVICE, non_blocking=True)
        second = second.to(DEVICE, non_blocking=True)
        fake_prob_0 = torch.softmax(model(first), 1)[:, 1]
        fake_prob_0_flip = torch.softmax(model(torch.flip(first, dim=-1)), 1)[:, 1]


        fake_prob_1 = torch.softmax(model(second), 1)[:, 1]
        fake_prob_1_flip = torch.softmax(model(torch.flip(second, dim=-1)), 1)[:, 1]
        predictions[indices.numpy()] = ((fake_prob_1 + fake_prob_1_flip)/2 > (fake_prob_0 + fake_prob_0_flip)/2).cpu().numpy()

submission = pd.DataFrame({'pair_id': query.pair_id, 'fake_position': predictions})
submission.to_csv('submission.csv', index=False)

with zipfile.ZipFile('submission.zip', 'w', zipfile.ZIP_DEFLATED) as archive:
    archive.write('submission.csv')

print(f'[dự đoán] {len(submission):,} cặp trong {time.time() - started:.1f}s')
print(f'[dự đoán] phân bố nhãn: {submission.fake_position.value_counts().to_dict()}')
print('[nộp bài] đã tạo submission.zip (chứa đúng submission.csv ở thư mục gốc)')